# Base Clients

> The core abstraction for different FL Clients. Any gneral client functionality resides here.

In [ ]:
#| default_exp clients

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fedai.clients import BaseClient
from fastcore.utils import *
from fastcore.all import *
import os
import networkx as nx
import pickle
import json
from collections import defaultdict, OrderedDict
from copy import deepcopy
import random
from enum import Enum
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from peft import *
from community import community_louvain
from fedai.utils import *
from fedai.client_selector import *
from fedai.optimizers import *
from tqdm import tqdm
import numpy as np
import pandas as pd
from loguru import logger
from fedai.utils import *
from fedai.metrics import *
from fedai.losses import *
from transformers import AutoTokenizer
from omegaconf.dictconfig import DictConfig
import numpy as np
import math
np.random.seed(42)
torch.manual_seed(42)

<torch._C.Generator>

In [ ]:
#| export
class AgentRole(Enum):
    SERVER = 1
    CLIENT = 2
    MARL = 3

## Per-FedAvg

Personalized Federated Learning with Theoretical Guarantees: A Model-Agnostic Meta-Learning Approach

In [ ]:
#| export
class PerAvgAgent(BaseClient):
    def __init__(self, 
                 id,
                 cfg,
                 state= None,
                 role= AgentRole.CLIENT,
                 block= None):
        
        super().__init__(id, cfg, state, role, block)
        
        if self.role == AgentRole.CLIENT:
            self.local_model = deepcopy(self.model)
            self.optimizer = PerFedAvgOpt(self.model.parameters(), lr=self.learning_rate)
            self.label_set = list(set(np.array([batch['y'] for batch in self.train_ds])))
            self.iter_trainloader = iter(self.train_loader)
            self.iter_testloader = iter(self.test_loader)

In [ ]:
#| export
@patch
def get_next_batch(self: PerAvgAgent, train= True) -> dict:
    
    loader_type = self.train_loader if train else self.test_loader
    to_iter = self.iter_trainloader if train else self.iter_testloader

    if(int(self.cfg.data.batch_size) == 0):
        for batch in loader_type:
            X = batch['x']
            y = batch['y']
            return (X.to(self.device), y.to(self.device))
        
    else:
        try:

            batch = next(to_iter)
            X = batch['x']
            y = batch['y']
            
        except StopIteration:

            if train:
                self.iter_trainloader = iter(self.train_loader)
                to_iter = self.iter_trainloader
            else:
                self.iter_testloader = iter(self.test_loader)
                to_iter = self.iter_testloader

            batch = next(to_iter)
            X = batch['x']
            y = batch['y']

            
        return (X.to(self.device), y.to(self.device))


### Training

In [ ]:
#| export
@patch
def _run_batch(self: PerAvgAgent) -> tuple:
    
    X, y = self.get_next_batch()
    self.optimizer.zero_grad()
    output = self.model(X)
    loss = self.loss(output, y)
    loss.backward()
    self.optimizer.step()

    X, y = self.get_next_batch()
    self.optimizer.zero_grad()
    output = self.model(X)
    loss = self.loss(output, y)
    loss.backward()
    self.optimizer.step(beta= self.cfg.beta)

    with torch.no_grad():
        for localweight, model_param in zip(self.local_model.parameters(), self.model):
            localweight.copy_(model_param)


    if self.cfg.model.grad_norm_clip:
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.model.grad_norm_clip)
    

    return loss, 

In [ ]:
#| export
@patch
def _run_epoch(self: PerAvgAgent):

    for i in range(self.num_minibatch):
        self._run_batch()

In [ ]:
#| export
@patch
def fit(self: PerAvgAgent) -> None:
    
    self.num_minibatch = int(len(self.train_ds) / int(self.cfg.data.batch_size))

    self.model = self.model.to(self.device)
    self.local_model = self.local_model.to(self.device)

    self.model.train()
    self.local_model.train()

    for _ in range(self.cfg.local_epochs):
        self._run_epoch()

    

In [ ]:
#| export
@patch
def train_one_step(self: PerAvgAgent) -> None:
    self.model.train()
    self.model = self.model.to(self.device)

    #step 1
    X, y = self.get_next_batch(train= False)
    self.optimizer.zero_grad()
    output = self.model(X)
    loss = self.loss(output, y)
    loss.backward()
    self.optimizer.step()
    
    #step 2
    X, y = self.get_next_batch(Train= True)
    self.optimizer.zero_grad()
    output = self.model(X)
    loss = self.loss(output, y)
    loss.backward()
    self.optimizer.step(beta=self.cfg.beta)

### Evaluation

In [ ]:
#| export
@patch
def evaluate(self: PerAvgAgent, t):
    self.cfg.agg ="mtl"
    lst_train_res = []
    lst_test_res = []
    for id in range(self.cfg.num_clients):
        
        client = self.client_fn(self.client_cls, self.cfg, id, self.latest_round, t, self.loss_fn, state_dir= self.cfg.state_dir)
        client.train_one_step()
        
        res_train = client.evaluate_local(loader= 'train')
        lst_train_res.append(res_train)

        res_test = client.evaluate_local(loader= 'test')
        lst_test_res.append(res_test)
        
    self.cfg.agg = "one_model"
    return lst_train_res, lst_test_res    


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()